In [1]:
!pip uninstall -y pyspark dataproc-spark-connect

Found existing installation: pyspark 4.0.2
Uninstalling pyspark-4.0.2:
  Successfully uninstalled pyspark-4.0.2
Found existing installation: dataproc-spark-connect 1.0.2
Uninstalling dataproc-spark-connect-1.0.2:
  Successfully uninstalled dataproc-spark-connect-1.0.2


In [2]:
!pip uninstall -y dataproc-spark-connect pyspark
!pip install -q pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.5 MB/s eta 0:00:00


In [3]:
import os
import pyspark
print("PySpark version:", pyspark.__version__)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CCF-Colab")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print(spark.version)

PySpark version: 3.5.1
3.5.1


In [4]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import time

In [5]:
data = [
    ("A", "B"),
    ("B", "C"),
    ("B", "D"),
    ("D", "E"),
    ("F", "G"),
    ("G", "H")
]

df_edges = spark.createDataFrame(data, ["src", "dst"])
df_edges.show(truncate=False)

+---+---+
|src|dst|
+---+---+
|A  |B  |
|B  |C  |
|B  |D  |
|D  |E  |
|F  |G  |
|G  |H  |
+---+---+



In [6]:
def make_undirected(df_edges: DataFrame) -> DataFrame:
    cleaned = (
        df_edges
        .filter(F.col("src").isNotNull() & F.col("dst").isNotNull())
        .filter(F.col("src") != F.col("dst"))
        .dropDuplicates()
    )

    reversed_df = cleaned.select(
        F.col("dst").alias("src"),
        F.col("src").alias("dst")
    )

    return cleaned.union(reversed_df).dropDuplicates()

In [7]:
df_undirected = make_undirected(df_edges).cache()
df_undirected.show(truncate=False)

+---+---+
|src|dst|
+---+---+
|G  |H  |
|B  |A  |
|G  |F  |
|B  |D  |
|A  |B  |
|F  |G  |
|H  |G  |
|E  |D  |
|B  |C  |
|C  |B  |
|D  |E  |
|D  |B  |
+---+---+



In [8]:
nodes_df = (
    df_undirected.select(F.col("src").alias("node"))
    .union(df_undirected.select(F.col("dst").alias("node")))
    .dropDuplicates()
    .cache()
)

nodes_df.show(truncate=False)

+----+
|node|
+----+
|G   |
|B   |
|C   |
|A   |
|E   |
|F   |
|H   |
|D   |
+----+



In [9]:
undirected_rdd = df_undirected.rdd.map(lambda r: (r["src"], r["dst"])).cache()
print(sorted(undirected_rdd.collect()))

[('A', 'B'), ('B', 'A'), ('B', 'C'), ('B', 'D'), ('C', 'B'), ('D', 'B'), ('D', 'E'), ('E', 'D'), ('F', 'G'), ('G', 'F'), ('G', 'H'), ('H', 'G')]


In [10]:
def run_ccf_rdd(df_undirected: DataFrame, max_iter: int = 100, verbose: bool = False):
    undirected_rdd = df_undirected.rdd.map(lambda r: (r["src"], r["dst"])).cache()
    undirected_rdd.count()

    graph_by_dst_rdd = undirected_rdd.map(lambda x: (x[1], x[0])).cache()
    graph_by_dst_rdd.count()

    labels_rdd = (
        undirected_rdd
        .flatMap(lambda x: [x[0], x[1]])
        .distinct()
        .map(lambda node: (node, node))
        .cache()
    )
    labels_rdd.count()

    changed = True
    iteration = 0

    while changed and iteration < max_iter:
        iteration += 1

        propagated = (
            graph_by_dst_rdd
            .join(labels_rdd)   # (dst, (src, label_of_dst))
            .map(lambda x: (x[1][0], x[1][1]))   # (src, neighbor_label)
        )

        min_neighbor_labels = propagated.reduceByKey(lambda a, b: a if a < b else b)

        next_labels_rdd = (
            labels_rdd
            .leftOuterJoin(min_neighbor_labels)
            .mapValues(lambda x: x[1] if x[1] is not None and x[1] < x[0] else x[0])
            .cache()
        )
        next_labels_rdd.count()

        num_changes = (
            labels_rdd
            .join(next_labels_rdd)
            .filter(lambda x: x[1][0] != x[1][1])
            .count()
        )

        if verbose:
            print(f"Iteration {iteration} -> changed labels: {num_changes}")

        changed = num_changes > 0

        labels_rdd.unpersist()
        labels_rdd = next_labels_rdd

    return labels_rdd, iteration

In [11]:
final_labels_rdd, rdd_iterations = run_ccf_rdd(df_undirected, max_iter=20, verbose=True)

print("RDD iterations =", rdd_iterations)
print(sorted(final_labels_rdd.collect(), key=lambda x: x[0]))

Iteration 1 -> changed labels: 6
Iteration 2 -> changed labels: 4
Iteration 3 -> changed labels: 1
Iteration 4 -> changed labels: 0
RDD iterations = 4
[('A', 'A'), ('B', 'A'), ('C', 'A'), ('D', 'A'), ('E', 'A'), ('F', 'F'), ('G', 'F'), ('H', 'F')]


In [12]:
spark.sparkContext.setCheckpointDir("/content/spark-checkpoints")

In [13]:
def run_ccf_dataframe(df_undirected: DataFrame, max_iter: int = 100, verbose: bool = False):
    nodes_df = (
        df_undirected.select(F.col("src").alias("node"))
        .union(df_undirected.select(F.col("dst").alias("node")))
        .dropDuplicates()
    )

    labels_df = (
        nodes_df
        .withColumn("label", F.col("node"))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    labels_df.count()

    changed = True
    iteration = 0

    while changed and iteration < max_iter:
        iteration += 1

        propagated = (
            df_undirected
            .join(labels_df, df_undirected["dst"] == labels_df["node"])
            .select(
                df_undirected["src"].alias("node"),
                labels_df["label"].alias("neighbor_label")
            )
        )

        min_neighbor_labels = (
            propagated
            .groupBy("node")
            .agg(F.min("neighbor_label").alias("min_neighbor_label"))
        )

        next_labels_raw = (
            labels_df
            .join(min_neighbor_labels, on="node", how="left")
            .select(
                F.col("node"),
                F.least(
                    F.col("label"),
                    F.coalesce(F.col("min_neighbor_label"), F.col("label"))
                ).alias("label")
            )
        )

        next_labels_df = next_labels_raw.localCheckpoint(eager=True)

        old_df = labels_df.alias("old")
        new_df = next_labels_df.alias("new")

        num_changes = (
            old_df
            .join(new_df, on="node")
            .filter(F.col("old.label") != F.col("new.label"))
            .count()
        )

        if verbose:
            print(f"Iteration {iteration} -> changed labels: {num_changes}")

        changed = num_changes > 0

        labels_df.unpersist()
        labels_df = next_labels_df.persist(StorageLevel.MEMORY_AND_DISK)
        labels_df.count()

    return labels_df, iteration

In [14]:
final_labels_df, df_iterations = run_ccf_dataframe(df_undirected, max_iter=20, verbose=True)

print("DataFrame iterations =", df_iterations)

(
    final_labels_df
    .select(F.col("node"), F.col("label").alias("component"))
    .orderBy("node")
    .show(truncate=False)
)

Iteration 1 -> changed labels: 6
Iteration 2 -> changed labels: 4
Iteration 3 -> changed labels: 1
Iteration 4 -> changed labels: 0
DataFrame iterations = 4
+----+---------+
|node|component|
+----+---------+
|A   |A        |
|B   |A        |
|C   |A        |
|D   |A        |
|E   |A        |
|F   |F        |
|G   |F        |
|H   |F        |
+----+---------+



In [15]:
def generate_multi_component_graph(num_components: int, component_size: int) -> DataFrame:
    edges = []
    for c in range(num_components):
        start = c * component_size + 1
        end = start + component_size - 1
        for i in range(start, end):
            edges.append((str(i), str(i + 1)))
    return spark.createDataFrame(edges, ["src", "dst"])

In [16]:
def benchmark_rdd(df_undirected: DataFrame):
    start = time.time()
    _, iterations = run_ccf_rdd(df_undirected, max_iter=100, verbose=False)
    duration_ms = int((time.time() - start) * 1000)
    return duration_ms, iterations

def benchmark_dataframe(df_undirected: DataFrame):
    start = time.time()
    _, iterations = run_ccf_dataframe(df_undirected, max_iter=100, verbose=False)
    duration_ms = int((time.time() - start) * 1000)
    return duration_ms, iterations

In [17]:
test_configs = [
    (5, 10),     # 50 nodes
    (5, 20),   # 100 nodes
    (10, 20),   # 200 nodes
    (10, 40)    # 400 nodes
]

In [ ]:
benchmark_results = []

for num_components, component_size in test_configs:
    print(f"Running config: components={num_components}, size={component_size}")

    df_graph = generate_multi_component_graph(num_components, component_size)
    df_undirected_bench = make_undirected(df_graph).cache()
    df_undirected_bench.count()

    num_nodes = num_components * component_size
    num_edges = df_graph.count()

    rdd_time_ms, rdd_iterations = benchmark_rdd(df_undirected_bench)
    df_time_ms, df_iterations = benchmark_dataframe(df_undirected_bench)

    benchmark_results.append((
        num_nodes,
        num_edges,
        rdd_time_ms,
        rdd_iterations,
        df_time_ms,
        df_iterations
    ))

    df_undirected_bench.unpersist()

benchmark_results

Running config: components=5, size=10


In [ ]:
results_df = spark.createDataFrame(
    benchmark_results,
    ["num_nodes", "num_edges", "rdd_time_ms", "rdd_iterations", "df_time_ms", "df_iterations"]
)

results_df.show(truncate=False)

In [ ]:
results_pd = results_df.toPandas()
results_pd

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(results_pd["num_nodes"], results_pd["rdd_time_ms"], marker="o", label="RDD")
plt.plot(results_pd["num_nodes"], results_pd["df_time_ms"], marker="o", label="DataFrame")
plt.xlabel("Number of nodes")
plt.ylabel("Runtime (ms)")
plt.title("RDD vs DataFrame runtime")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_pd["num_nodes"], results_pd["rdd_iterations"], marker="o", label="RDD iterations")
plt.plot(results_pd["num_nodes"], results_pd["df_iterations"], marker="o", label="DataFrame iterations")
plt.xlabel("Number of nodes")
plt.ylabel("Iterations")
plt.title("RDD vs DataFrame iterations")
plt.legend()
plt.grid(True)
plt.show()